# 鼓点计数参考题解：从整段回归到事件检测

本题最终只评估每段音频里 `KD` 和 `SD` 的数量，但训练集其实给了更强的监督：`annotation_xml` 中有精确的敲击时间。

因此这份题解不再把整段梅尔频谱当成一张图直接回归两个数字，而是分成三步：

1. 先把问题改写成帧级鼓点检测。
2. 再利用 `audio_training_hits` 和 `audio_isolated` 做小数据增强。
3. 最后把模型输出的时间概率曲线转成峰值个数，也就得到了计数结果。

## 1. baseline 为什么不够

baseline 的做法是：

- 先把整段音频转成梅尔频谱。
- 再用一个 CNN 把整段频谱压成全局向量。
- 最后直接回归 `KD` 和 `SD` 数量。

这会丢掉一个关键结构：**鼓点计数本质上来自时间轴上的离散事件数**。

只要把时间维度在中间过早做全局池化，模型就很难真正学到“哪里发生了一次鼓点”和“多少个峰值对应多少次敲击”。

所以本题更自然的建模方式是：

> 先做 onset detection，再做 peak counting。

In [ ]:
from __future__ import annotations

import copy
import functools
import json
import math
import random
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy import signal

# 题目里一共会出现三类鼓点，其中 HH 只作为辅助任务，不进入最终提交。
INSTRUMENTS = ('KD', 'SD', 'HH')
COUNT_INSTRUMENTS = ('KD', 'SD')


@dataclass
class AudioConfig:
    sample_rate: int = 22050
    n_fft: int = 1024
    hop_length: int = 128
    n_mels: int = 96
    fmin: float = 20.0
    fmax: float = 11025.0
    gaussian_sigma_frames: float = 1.0
    target_radius_frames: int = 2


@dataclass
class AugmentConfig:
    synth_prob: float = 0.25
    synth_background_gain: float = 0.04
    jitter_ms: float = 8.0
    gain_low: float = 0.85
    gain_high: float = 1.15
    stretch_low: float = 0.97
    stretch_high: float = 1.03
    real_gain_low: float = 0.90
    real_gain_high: float = 1.10


@dataclass
class TrainConfig:
    seed: int = 3407
    folds: int = 5
    batch_size: int = 8
    num_workers: int = 0
    pretrain_epochs: int = 6
    finetune_epochs: int = 20
    pretrain_steps_per_epoch: int = 256
    train_steps_per_epoch: int = 384
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    crop_seconds_min: float = 8.0
    crop_seconds_max: float = 12.0
    patience: int = 5
    kd_default_threshold: float = 0.40
    sd_default_threshold: float = 0.42
    kd_default_distance: int = 8
    sd_default_distance: int = 8
    threshold_grid: tuple[float, ...] = (0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60)
    distance_grid: tuple[int, ...] = (6, 8, 10, 12)
    class_weights: tuple[float, float, float] = (1.6, 1.6, 0.6)
    pos_weights: tuple[float, float, float] = (6.0, 6.0, 3.0)
    amp_enabled: bool = True


@dataclass
class Config:
    # 如果在 Kaggle 中使用别的挂载路径，只需要改这两个路径即可。
    dataset_dir: str = 'dataset'
    output_dir: str = 'artifacts/refresult1'
    audio: AudioConfig = field(default_factory=AudioConfig)
    augment: AugmentConfig = field(default_factory=AugmentConfig)
    train: TrainConfig = field(default_factory=TrainConfig)


@dataclass
class DrumExample:
    stem: str
    subset: str
    audio_path: str
    duration_sec: float
    onsets: dict[str, list[float]]
    counts: dict[str, int]
    xml_path: str | None = None


@dataclass
class HitSnippet:
    instrument: str
    source: str
    waveform: np.ndarray
    duration_sec: float
    rms: float


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


config = Config()
set_seed(config.train.seed)

# Kaggle T4 上通常是 cuda，本地调试时可能是 cpu。
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset_dir = Path(config.dataset_dir)

if hasattr(torch, 'set_float32_matmul_precision'):
    torch.set_float32_matmul_precision('high')

print(device)
print(dataset_dir.resolve())
print(Path(config.output_dir).resolve())

## 2. 数据读取与帧级标签

这里做两件事：

- 把所有音频统一成单声道、`22.05kHz`。
- 把 XML 中的 `KD / SD / HH` onset 映射到帧级热图。

虽然最终只提交 `KD` 和 `SD`，但我额外保留 `HH` 作为辅助任务。原因是踩镲经常和军鼓在高频上互相干扰，显式预测 `HH` 往往能减轻 `SD` 的误检。

In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def clean_artifact_dir(path: Path) -> None:
    # 每次重新训练前都清理旧产物，避免不同实验的文件互相污染。
    ensure_dir(path)
    patterns = (
        'fold_*.pt',
        'submission*.csv',
        'submission.zip',
        'oof_*.json',
        'oof_*.csv',
        'peak_params.json',
        'fold_scores.json',
        'manifest.json',
    )
    for pattern in patterns:
        for file_path in path.glob(pattern):
            file_path.unlink()


def save_json(data: Any, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False))


def read_audio(path: str | Path, sample_rate: int) -> np.ndarray:
    # 统一转成单声道，并重采样到固定采样率。
    waveform, sr = sf.read(str(path), always_2d=False)
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    if sr != sample_rate:
        tensor = torch.from_numpy(waveform)
        waveform = torchaudio.functional.resample(tensor, orig_freq=sr, new_freq=sample_rate).numpy()
    peak = np.max(np.abs(waveform)) + 1e-6
    waveform = waveform / peak
    return waveform.astype(np.float32)


def audio_duration_seconds(path: str | Path) -> float:
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate)


def parse_annotation(xml_path: str | Path) -> tuple[dict[str, list[float]], dict[str, int]]:
    # 解析 XML，保留每类鼓点的精确 onset 时间。
    root = ET.parse(str(xml_path)).getroot()
    onsets = {inst: [] for inst in INSTRUMENTS}
    for event in root.findall('.//event'):
        instrument = event.findtext('instrument')
        onset_text = event.findtext('onsetSec')
        if instrument in onsets and onset_text is not None:
            onsets[instrument].append(max(0.0, float(onset_text)))
    for values in onsets.values():
        values.sort()
    counts = {inst: len(onsets[inst]) for inst in INSTRUMENTS}
    return onsets, counts


def list_audio_files(directory: Path) -> list[Path]:
    return sorted(path for path in directory.iterdir() if path.suffix == '.wav' and not path.name.startswith('.'))


@functools.lru_cache(maxsize=8)
def build_feature_transforms(sample_rate: int, n_fft: int, hop_length: int, n_mels: int, fmin: float, fmax: float):
    # 这里显式保留两个通道：log-mel 和正向时间差分。
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate,
        n_fft=n_fft,
        hop_length=hop_length,
        n_mels=n_mels,
        f_min=fmin,
        f_max=fmax,
        power=2.0,
        center=False,
        norm='slaney',
        mel_scale='htk',
    )
    db_transform = torchaudio.transforms.AmplitudeToDB(stype='power')
    return mel_transform, db_transform


def compute_features(waveform: np.ndarray, audio_cfg: AudioConfig) -> np.ndarray:
    # 第一通道：标准 log-mel。
    # 第二通道：只保留正向变化，显式强调瞬态 onset。
    mel_transform, db_transform = build_feature_transforms(
        audio_cfg.sample_rate,
        audio_cfg.n_fft,
        audio_cfg.hop_length,
        audio_cfg.n_mels,
        audio_cfg.fmin,
        audio_cfg.fmax,
    )
    with torch.no_grad():
        tensor = torch.from_numpy(waveform).unsqueeze(0)
        mel = mel_transform(tensor)
        log_mel = db_transform(mel).squeeze(0).cpu().numpy()
    delta = np.diff(log_mel, axis=1, prepend=log_mel[:, :1])
    pos_delta = np.clip(delta, a_min=0.0, a_max=None)
    log_mel = (log_mel - np.mean(log_mel)) / (np.std(log_mel) + 1e-6)
    pos_delta = pos_delta / (np.std(pos_delta) + 1e-6)
    return np.stack([log_mel, pos_delta], axis=0).astype(np.float32)


def seconds_to_frames(seconds: np.ndarray | list[float], audio_cfg: AudioConfig) -> np.ndarray:
    seconds_array = np.asarray(seconds, dtype=np.float32)
    return np.round(seconds_array * audio_cfg.sample_rate / audio_cfg.hop_length).astype(np.int64)


def build_target_heatmap(frame_count: int, onsets: dict[str, list[float]], audio_cfg: AudioConfig) -> np.ndarray:
    # 把精确 onset 转成一个小高斯峰，而不是只有单个硬标签点。
    target = np.zeros((len(INSTRUMENTS), frame_count), dtype=np.float32)
    for instrument_index, instrument in enumerate(INSTRUMENTS):
        frames = seconds_to_frames(onsets[instrument], audio_cfg)
        for frame in frames:
            if frame_count <= 0:
                break
            center = int(np.clip(frame, 0, frame_count - 1))
            target[instrument_index, center] = 1.0
            for delta in range(1, audio_cfg.target_radius_frames + 1):
                weight = math.exp(-(delta ** 2) / (2.0 * audio_cfg.gaussian_sigma_frames ** 2))
                left = center - delta
                right = center + delta
                if left >= 0:
                    target[instrument_index, left] = max(target[instrument_index, left], weight)
                if right < frame_count:
                    target[instrument_index, right] = max(target[instrument_index, right], weight)
    return target


def build_examples(dataset_root: Path):
    def build_train_examples() -> list[DrumExample]:
        audio_dir = dataset_root / 'training_set' / 'audio_mix'
        xml_dir = dataset_root / 'training_set' / 'annotation_xml'
        examples = []
        for audio_path in list_audio_files(audio_dir):
            xml_path = xml_dir / f'{audio_path.stem}.xml'
            onsets, counts = parse_annotation(xml_path)
            examples.append(
                DrumExample(
                    stem=audio_path.stem.replace('_MIX', ''),
                    subset='train',
                    audio_path=str(audio_path),
                    duration_sec=audio_duration_seconds(audio_path),
                    onsets=onsets,
                    counts=counts,
                    xml_path=str(xml_path),
                )
            )
        return examples

    def build_eval_examples(subset_name: str) -> list[DrumExample]:
        audio_dir = dataset_root / subset_name / 'audio_mix'
        examples = []
        for audio_path in list_audio_files(audio_dir):
            examples.append(
                DrumExample(
                    stem=audio_path.stem.replace('_MIX', ''),
                    subset=subset_name,
                    audio_path=str(audio_path),
                    duration_sec=audio_duration_seconds(audio_path),
                    onsets={inst: [] for inst in INSTRUMENTS},
                    counts={inst: 0 for inst in INSTRUMENTS},
                )
            )
        return examples

    train_examples = build_train_examples()
    validation_examples = build_eval_examples('validation_set')
    testing_examples = build_eval_examples('testing_set')
    return train_examples, validation_examples, testing_examples

## 3. 用训练素材构造 hit bank

数据量很小，直接端到端训练很容易过拟合。

这里不做“随便切块”，而是先把训练集中提供的辅助音频整理成一个可复用的 `hit bank`：

- `audio_training_hits`：主要来源。虽然文件名叫 training hits，但它们并不是一个个独立小 wav，而是同类鼓点连续排列的长音频，所以要先做能量分段。
- `audio_isolated`：补充来源。它覆盖不完整，但和原始 onset 对齐，适合按标签时间直接切出更干净的单击片段。

有了 `hit bank` 之后，就可以沿用真实节奏模板重新合成训练样本。这样增强出来的样本仍然带有精确标签。

In [ ]:
def build_rms_envelope(waveform: np.ndarray, sample_rate: int, frame_ms: float = 10.0):
    # 先把长音频切成粗粒度能量帧，后面用来分段。
    frame_size = max(1, int(round(sample_rate * frame_ms / 1000.0)))
    if waveform.shape[0] < frame_size:
        waveform = np.pad(waveform, (0, frame_size - waveform.shape[0]))
    pad = (-waveform.shape[0]) % frame_size
    if pad:
        waveform = np.pad(waveform, (0, pad))
    frames = waveform.reshape(-1, frame_size)
    envelope = np.sqrt(np.mean(np.square(frames), axis=1) + 1e-10)
    return envelope.astype(np.float32), frame_size


def extract_hit_segments_from_bank(audio_path: Path, instrument: str, sample_rate: int) -> list[HitSnippet]:
    # 这个函数负责从 *_train.wav 中自动分离出一个个单击片段。
    waveform = read_audio(audio_path, sample_rate=sample_rate)
    envelope, frame_size = build_rms_envelope(waveform, sample_rate=sample_rate)
    if envelope.size == 0:
        return []

    peak = float(envelope.max())
    floor = float(np.median(envelope))
    threshold = max(peak * 0.18, floor * 2.5)
    active = envelope > threshold

    # 如果整段都很弱，就至少保留最大峰附近，避免空库。
    if not active.any():
        peak_index = int(np.argmax(envelope))
        active[max(0, peak_index - 1): peak_index + 2] = True

    # 小范围桥接，把非常短的静音缺口合并掉。
    silence_bridge_frames = 2
    for index in range(1, len(active) - silence_bridge_frames):
        if active[index - 1] and not active[index] and active[index:index + silence_bridge_frames].any():
            active[index:index + silence_bridge_frames] = True

    segments = []
    start = None
    for index, is_active in enumerate(active.tolist() + [False]):
        if is_active and start is None:
            start = index
        elif not is_active and start is not None:
            end = index
            start_sample = max(0, start * frame_size - int(0.015 * sample_rate))
            end_sample = min(len(waveform), end * frame_size + int(0.080 * sample_rate))
            snippet = waveform[start_sample:end_sample]
            duration_sec = snippet.shape[0] / sample_rate
            if 0.02 <= duration_sec <= 0.70:
                rms = float(np.sqrt(np.mean(np.square(snippet)) + 1e-10))
                segments.append(
                    HitSnippet(
                        instrument=instrument,
                        source=audio_path.name,
                        waveform=snippet.astype(np.float32),
                        duration_sec=duration_sec,
                        rms=rms,
                    )
                )
            start = None
    return segments


def fixed_window_seconds(instrument: str) -> tuple[float, float]:
    # 不同鼓点的余响长度不同，所以切窗长度也稍微区分。
    if instrument == 'HH':
        return 0.01, 0.14
    if instrument == 'KD':
        return 0.02, 0.30
    return 0.02, 0.34


def extract_hit_segments_from_isolated(isolated_path: Path, example: DrumExample, instrument: str, sample_rate: int) -> list[HitSnippet]:
    # 这部分不做自动分段，而是直接按真实标签时间切取。
    waveform = read_audio(isolated_path, sample_rate=sample_rate)
    pre_sec, post_sec = fixed_window_seconds(instrument)
    snippets = []
    for onset in example.onsets[instrument]:
        start = max(0, int(round((onset - pre_sec) * sample_rate)))
        end = min(len(waveform), int(round((onset + post_sec) * sample_rate)))
        if end <= start:
            continue
        snippet = waveform[start:end].astype(np.float32)
        rms = float(np.sqrt(np.mean(np.square(snippet)) + 1e-10))
        if rms < 1e-4:
            continue
        snippets.append(
            HitSnippet(
                instrument=instrument,
                source=isolated_path.name,
                waveform=snippet,
                duration_sec=snippet.shape[0] / sample_rate,
                rms=rms,
            )
        )
    return snippets


def build_hit_bank(dataset_root: Path, train_examples: list[DrumExample], audio_cfg: AudioConfig) -> dict[str, list[HitSnippet]]:
    bank = {inst: [] for inst in INSTRUMENTS}

    # 1) 先从 training hits 中挖出大量单击片段。
    hits_dir = dataset_root / 'training_set' / 'audio_training_hits'
    for path in list_audio_files(hits_dir):
        instrument = path.stem.split('_')[-2]
        if instrument in bank:
            bank[instrument].extend(extract_hit_segments_from_bank(path, instrument, audio_cfg.sample_rate))

    # 2) 再从 isolated 轨道中补充更干净的对齐样本。
    example_by_stem = {example.stem: example for example in train_examples}
    isolated_dir = dataset_root / 'training_set' / 'audio_isolated'
    for path in list_audio_files(isolated_dir):
        stem, instrument = path.stem.rsplit('_', 1)
        if instrument in bank and stem in example_by_stem:
            bank[instrument].extend(
                extract_hit_segments_from_isolated(path, example_by_stem[stem], instrument, audio_cfg.sample_rate)
            )

    return bank


def choose_hit(hit_bank: dict[str, list[HitSnippet]], instrument: str, rng: random.Random) -> HitSnippet:
    candidates = hit_bank[instrument]
    if not candidates:
        raise ValueError(f'hit bank for {instrument} is empty')
    return candidates[rng.randrange(len(candidates))]


def waveform_pool(examples: list[DrumExample], sample_rate: int) -> list[np.ndarray]:
    return [read_audio(example.audio_path, sample_rate) for example in examples]


def maybe_add_background(waveform: np.ndarray, background_pool: list[np.ndarray], gain: float, rng: random.Random) -> np.ndarray:
    # 合成样本容易太“干净”，所以叠加一个很弱的真实背景床。
    if not background_pool or gain <= 0.0:
        return waveform
    background = background_pool[rng.randrange(len(background_pool))]
    if background.shape[0] >= waveform.shape[0]:
        max_start = background.shape[0] - waveform.shape[0]
        start = rng.randint(0, max_start) if max_start > 0 else 0
        patch = background[start:start + waveform.shape[0]]
    else:
        patch = np.zeros_like(waveform)
        patch[:background.shape[0]] = background
    return np.clip(waveform + gain * patch, -1.0, 1.0).astype(np.float32)


def time_stretch_hit(waveform: np.ndarray, stretch_factor: float) -> np.ndarray:
    # 这里不做大幅度的速度变化，只允许很轻微的长度扰动。
    if abs(stretch_factor - 1.0) < 1e-4 or waveform.shape[0] < 4:
        return waveform
    target_length = max(1, int(round(waveform.shape[0] * stretch_factor)))
    return signal.resample(waveform, target_length).astype(np.float32)


def synthesize_waveform_from_template(template_example: DrumExample, hit_bank: dict[str, list[HitSnippet]], audio_cfg: AudioConfig, augment_cfg: AugmentConfig, background_pool: list[np.ndarray], rng: random.Random):
    # 核心思想：复用真实节奏模板，只替换每个时间点上的音色。
    total_samples = int(round(template_example.duration_sec * audio_cfg.sample_rate))
    waveform = np.zeros(total_samples, dtype=np.float32)
    synth_onsets = {inst: [] for inst in INSTRUMENTS}
    jitter_limit = augment_cfg.jitter_ms / 1000.0

    for instrument in INSTRUMENTS:
        for onset in template_example.onsets[instrument]:
            snippet = choose_hit(hit_bank, instrument, rng)
            segment = time_stretch_hit(snippet.waveform.copy(), rng.uniform(augment_cfg.stretch_low, augment_cfg.stretch_high))
            segment = rng.uniform(augment_cfg.gain_low, augment_cfg.gain_high) * segment
            shifted_onset = float(np.clip(onset + rng.uniform(-jitter_limit, jitter_limit), 0.0, template_example.duration_sec))
            synth_onsets[instrument].append(shifted_onset)
            start_sample = int(round(shifted_onset * audio_cfg.sample_rate))
            end_sample = min(total_samples, start_sample + segment.shape[0])
            if end_sample <= start_sample:
                continue
            valid = end_sample - start_sample
            waveform[start_sample:end_sample] += segment[:valid]

    waveform = np.clip(waveform, -1.0, 1.0)
    waveform = maybe_add_background(waveform, background_pool, augment_cfg.synth_background_gain, rng)
    for values in synth_onsets.values():
        values.sort()
    return waveform.astype(np.float32), synth_onsets


def crop_waveform(waveform: np.ndarray, onsets: dict[str, list[float]], sample_rate: int, crop_seconds: float, rng: random.Random):
    # 训练时做随机裁剪，提高不同局部片段的覆盖率。
    crop_samples = int(round(crop_seconds * sample_rate))
    total_samples = waveform.shape[0]
    if total_samples >= crop_samples:
        max_start = total_samples - crop_samples
        start_sample = rng.randint(0, max_start) if max_start > 0 else 0
        end_sample = start_sample + crop_samples
        cropped = waveform[start_sample:end_sample]
        start_sec = start_sample / sample_rate
        end_sec = end_sample / sample_rate
        cropped_onsets = {
            inst: [onset - start_sec for onset in values if start_sec <= onset < end_sec]
            for inst, values in onsets.items()
        }
        return cropped.astype(np.float32), cropped_onsets
    padded = np.zeros(crop_samples, dtype=np.float32)
    padded[:total_samples] = waveform
    return padded, copy.deepcopy(onsets)


def split_indices(num_items: int, folds: int, seed: int):
    indices = list(range(num_items))
    rng = random.Random(seed)
    rng.shuffle(indices)
    fold_slices = np.array_split(np.array(indices), folds)
    result = []
    for fold_index in range(folds):
        valid = fold_slices[fold_index].tolist()
        train = [item for index, fold in enumerate(fold_slices) if index != fold_index for item in fold.tolist()]
        result.append((train, valid))
    return result

## 4. 训练样本如何生成

训练时我混合两类样本：

- 真实混音片段的随机裁剪。
- 根据真实节奏模板合成的新片段。

合成时的原则很简单：

1. 随机选一首训练样本，读取它的真实 `KD / SD / HH` 时间轴。
2. 从 `hit bank` 中为每个事件采样一个单击片段。
3. 对每个单击施加轻微的增益扰动、时长扰动和时间抖动。
4. 可选叠加很弱的真实背景床，避免合成域过于干净。

这样做的目的不是“凭空编节奏”，而是 **在不破坏标签正确性的前提下增加音色变化**。

In [ ]:
class DrumTrainDataset(torch.utils.data.Dataset):
    def __init__(self, examples: list[DrumExample], hit_bank: dict[str, list[HitSnippet]], background_pool: list[np.ndarray], audio_cfg: AudioConfig, augment_cfg: AugmentConfig, train_cfg: TrainConfig, steps_per_epoch: int, seed: int, synth_only: bool = False):
        self.examples = examples
        self.hit_bank = hit_bank
        self.background_pool = background_pool
        self.audio_cfg = audio_cfg
        self.augment_cfg = augment_cfg
        self.train_cfg = train_cfg
        self.steps_per_epoch = steps_per_epoch
        self.synth_only = synth_only
        self.rng = random.Random(seed)
        self.waveforms = {example.stem: read_audio(example.audio_path, audio_cfg.sample_rate) for example in examples}

    def __len__(self) -> int:
        return self.steps_per_epoch

    def __getitem__(self, index: int) -> dict[str, Any]:
        use_synth = self.synth_only or (self.rng.random() < self.augment_cfg.synth_prob)
        template_example = self.examples[self.rng.randrange(len(self.examples))]

        if use_synth:
            waveform, onsets = synthesize_waveform_from_template(
                template_example,
                self.hit_bank,
                self.audio_cfg,
                self.augment_cfg,
                self.background_pool,
                self.rng,
            )
        else:
            waveform = self.waveforms[template_example.stem].copy()
            waveform = np.clip(
                self.rng.uniform(self.augment_cfg.real_gain_low, self.augment_cfg.real_gain_high) * waveform,
                -1.0,
                1.0,
            ).astype(np.float32)
            onsets = copy.deepcopy(template_example.onsets)

        crop_seconds = self.rng.uniform(self.train_cfg.crop_seconds_min, self.train_cfg.crop_seconds_max)
        waveform, onsets = crop_waveform(waveform, onsets, self.audio_cfg.sample_rate, crop_seconds, self.rng)
        features = compute_features(waveform, self.audio_cfg)
        targets = build_target_heatmap(features.shape[-1], onsets, self.audio_cfg)
        return {'features': features, 'targets': targets, 'name': template_example.stem}


class DrumEvalDataset(torch.utils.data.Dataset):
    def __init__(self, examples: list[DrumExample], audio_cfg: AudioConfig):
        self.examples = examples
        self.audio_cfg = audio_cfg

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int) -> dict[str, Any]:
        example = self.examples[index]
        waveform = read_audio(example.audio_path, self.audio_cfg.sample_rate)
        features = compute_features(waveform, self.audio_cfg)
        return {'features': features, 'name': example.stem, 'counts': example.counts}


def collate_train(batch: list[dict[str, Any]]) -> dict[str, Any]:
    # 同一 batch 内的样本长度可能不同，因此需要按时间维补齐。
    max_time = max(item['features'].shape[-1] for item in batch)
    batch_size = len(batch)
    channels = batch[0]['features'].shape[0]
    mels = batch[0]['features'].shape[1]
    target_channels = batch[0]['targets'].shape[0]

    features = torch.zeros((batch_size, channels, mels, max_time), dtype=torch.float32)
    targets = torch.zeros((batch_size, target_channels, max_time), dtype=torch.float32)
    mask = torch.zeros((batch_size, max_time), dtype=torch.float32)
    names = []

    for index, item in enumerate(batch):
        time_steps = item['features'].shape[-1]
        features[index, :, :, :time_steps] = torch.from_numpy(item['features'])
        targets[index, :, :time_steps] = torch.from_numpy(item['targets'])
        mask[index, :time_steps] = 1.0
        names.append(item['name'])

    return {'features': features, 'targets': targets, 'mask': mask, 'names': names}


def collate_eval(batch: list[dict[str, Any]]) -> dict[str, Any]:
    max_time = max(item['features'].shape[-1] for item in batch)
    batch_size = len(batch)
    channels = batch[0]['features'].shape[0]
    mels = batch[0]['features'].shape[1]

    features = torch.zeros((batch_size, channels, mels, max_time), dtype=torch.float32)
    mask = torch.zeros((batch_size, max_time), dtype=torch.float32)
    names = []
    counts = []

    for index, item in enumerate(batch):
        time_steps = item['features'].shape[-1]
        features[index, :, :, :time_steps] = torch.from_numpy(item['features'])
        mask[index, :time_steps] = 1.0
        names.append(item['name'])
        counts.append(item['counts'])

    return {'features': features, 'mask': mask, 'names': names, 'counts': counts}

## 5. 模型：保留时间轴的小型 CRNN

模型不需要很大，但有两个要求：

- 频率维可以逐步压缩。
- 时间维不能过早丢掉。

因此这里用一个比较轻的 `CRNN`：

- 前端 2D CNN 只沿频率轴做池化。
- 后端双向 GRU 在时间维上整合上下文。
- 输出 `KD / SD / HH` 三条帧级概率曲线。

它比“整段池化再回归”更贴合任务，也比上重型检测网络更容易讲清楚。

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.block(inputs)


class DrumCRNN(nn.Module):
    def __init__(self, n_mels: int):
        super().__init__()

        # 这里只沿频率轴做池化，尽量保留时间分辨率。
        self.encoder = nn.Sequential(
            ConvBlock(2, 32),
            nn.MaxPool2d(kernel_size=(2, 1)),
            ConvBlock(32, 64),
            nn.MaxPool2d(kernel_size=(2, 1)),
            ConvBlock(64, 96),
            nn.MaxPool2d(kernel_size=(2, 1)),
            ConvBlock(96, 128),
        )

        reduced_mels = n_mels // 8
        self.sequence_projection = nn.Linear(128 * reduced_mels, 192)
        self.rnn = nn.GRU(
            input_size=192,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(256),
            nn.Dropout(0.2),
            nn.Linear(256, len(INSTRUMENTS)),
        )

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        encoded = self.encoder(inputs)
        encoded = encoded.permute(0, 3, 1, 2).contiguous()
        batch, time_steps, channels, freqs = encoded.shape
        encoded = encoded.view(batch, time_steps, channels * freqs)
        encoded = self.sequence_projection(encoded)
        encoded, _ = self.rnn(encoded)
        outputs = self.head(encoded)
        return outputs.transpose(1, 2)


def create_grad_scaler(device: torch.device, enabled: bool):
    if device.type != 'cuda' or not enabled:
        return None
    return torch.amp.GradScaler('cuda', enabled=True)


def masked_bce_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor, class_weights: torch.Tensor, pos_weights: torch.Tensor) -> torch.Tensor:
    # 由于 onset 标签非常稀疏，需要对正样本额外加权。
    loss = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction='none',
        pos_weight=pos_weights.view(1, -1, 1),
    )
    loss = loss * class_weights.view(1, -1, 1)
    loss = loss * mask.unsqueeze(1)
    normalizer = mask.sum() * targets.shape[1]
    return loss.sum() / normalizer.clamp_min(1.0)

## 6. 从帧级概率到最终计数

训练目标是帧级热图，但评测指标是片段级数量。

因此推理阶段要再做一步：把 `KD / SD` 的概率曲线变成峰值数量。流程是：

1. 先对概率曲线做轻微平滑。
2. 再做 `find_peaks`。
3. 峰的个数就是对应乐器的计数。

这里的阈值和最小峰距不能手拍脑袋设，所以我用 OOF 预测结果做一个小网格搜索，把 `KD` 和 `SD` 的规则分别调好再固定下来。

In [ ]:
def run_epoch(model: nn.Module, loader: torch.utils.data.DataLoader, optimizer: torch.optim.Optimizer, device: torch.device, class_weights: torch.Tensor, pos_weights: torch.Tensor, scaler, amp_enabled: bool) -> float:
    model.train()
    losses = []
    autocast_enabled = amp_enabled and device.type == 'cuda'

    for batch in loader:
        features = batch['features'].to(device)
        targets = batch['targets'].to(device)
        mask = batch['mask'].to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=autocast_enabled):
            logits = model(features)
            loss = masked_bce_loss(logits, targets, mask, class_weights, pos_weights)

        if scaler is None:
            loss.backward()
            optimizer.step()
        else:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        losses.append(float(loss.detach().cpu()))

    return float(np.mean(losses)) if losses else 0.0


def smooth_track(track: np.ndarray) -> np.ndarray:
    kernel = np.array([0.15, 0.70, 0.15], dtype=np.float32)
    return np.convolve(track, kernel, mode='same')


def count_from_track(track: np.ndarray, threshold: float, min_distance: int) -> int:
    smoothed = smooth_track(track)
    peaks, _ = signal.find_peaks(smoothed, height=threshold, distance=min_distance)
    return int(len(peaks))


def clip_mae(predictions: list[tuple[int, int]], examples: list[DrumExample]) -> float:
    errors = []
    for (pred_kd, pred_sd), example in zip(predictions, examples):
        errors.append((abs(pred_kd - example.counts['KD']) + abs(pred_sd - example.counts['SD'])) / 2.0)
    return float(np.mean(errors)) if errors else 0.0


@torch.no_grad()
def predict_probabilities(model: nn.Module, loader: torch.utils.data.DataLoader, device: torch.device) -> dict[str, np.ndarray]:
    model.eval()
    predictions = {}

    for batch in loader:
        features = batch['features'].to(device)
        mask = batch['mask'].to(device)
        logits = model(features)
        probs = torch.sigmoid(logits).cpu().numpy()
        lengths = mask.sum(dim=1).cpu().numpy().astype(np.int64)

        for row_index, name in enumerate(batch['names']):
            valid_length = int(lengths[row_index])
            predictions[name] = probs[row_index, :, :valid_length].astype(np.float32)

    return predictions


def evaluate_with_default_thresholds(probs_by_name: dict[str, np.ndarray], examples: list[DrumExample], train_cfg: TrainConfig) -> float:
    predictions = []
    for example in examples:
        probs = probs_by_name[example.stem]
        kd_count = count_from_track(probs[0], train_cfg.kd_default_threshold, train_cfg.kd_default_distance)
        sd_count = count_from_track(probs[1], train_cfg.sd_default_threshold, train_cfg.sd_default_distance)
        predictions.append((kd_count, sd_count))
    return clip_mae(predictions, examples)


def tune_peak_parameters(probs_by_name: dict[str, np.ndarray], examples: list[DrumExample], train_cfg: TrainConfig):
    best = {}
    for instrument_index, instrument in enumerate(COUNT_INSTRUMENTS):
        best_error = float('inf')
        if instrument == 'KD':
            best_params = {'threshold': train_cfg.kd_default_threshold, 'min_distance': train_cfg.kd_default_distance}
        else:
            best_params = {'threshold': train_cfg.sd_default_threshold, 'min_distance': train_cfg.sd_default_distance}

        for threshold in train_cfg.threshold_grid:
            for min_distance in train_cfg.distance_grid:
                errors = []
                for example in examples:
                    probs = probs_by_name[example.stem][instrument_index]
                    pred = count_from_track(probs, float(threshold), int(min_distance))
                    errors.append(abs(pred - example.counts[instrument]))
                mae = float(np.mean(errors))
                if mae < best_error:
                    best_error = mae
                    best_params = {'threshold': float(threshold), 'min_distance': int(min_distance)}

        best[instrument] = best_params
    return best


def train_shared_initialization(examples: list[DrumExample], hit_bank: dict[str, list[HitSnippet]], background_pool: list[np.ndarray], config: Config, device: torch.device):
    # 先用纯合成样本做一个共享初始化，让模型先学会“鼓点长什么样”。
    dataset = DrumTrainDataset(
        examples,
        hit_bank,
        background_pool,
        config.audio,
        config.augment,
        config.train,
        config.train.pretrain_steps_per_epoch,
        config.train.seed,
        synth_only=True,
    )
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=config.train.batch_size,
        shuffle=False,
        num_workers=config.train.num_workers,
        collate_fn=collate_train,
    )
    model = DrumCRNN(config.audio.n_mels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.train.learning_rate, weight_decay=config.train.weight_decay)
    scaler = create_grad_scaler(device, config.train.amp_enabled)
    class_weights = torch.tensor(config.train.class_weights, dtype=torch.float32, device=device)
    pos_weights = torch.tensor(config.train.pos_weights, dtype=torch.float32, device=device)

    for _ in range(config.train.pretrain_epochs):
        run_epoch(model, loader, optimizer, device, class_weights, pos_weights, scaler, config.train.amp_enabled)

    return copy.deepcopy(model.state_dict())


def train_fold(fold_index: int, train_examples: list[DrumExample], valid_examples: list[DrumExample], hit_bank: dict[str, list[HitSnippet]], background_pool: list[np.ndarray], initial_state: dict[str, torch.Tensor], config: Config, device: torch.device, output_dir: Path):
    model = DrumCRNN(config.audio.n_mels).to(device)
    model.load_state_dict(initial_state)

    train_dataset = DrumTrainDataset(
        train_examples,
        hit_bank,
        background_pool,
        config.audio,
        config.augment,
        config.train,
        config.train.train_steps_per_epoch,
        config.train.seed + fold_index,
    )
    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config.train.batch_size,
        shuffle=False,
        num_workers=config.train.num_workers,
        collate_fn=collate_train,
    )
    valid_loader = torch.utils.data.DataLoader(
        DrumEvalDataset(valid_examples, config.audio),
        batch_size=1,
        shuffle=False,
        num_workers=config.train.num_workers,
        collate_fn=collate_eval,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.train.learning_rate, weight_decay=config.train.weight_decay)
    scaler = create_grad_scaler(device, config.train.amp_enabled)
    class_weights = torch.tensor(config.train.class_weights, dtype=torch.float32, device=device)
    pos_weights = torch.tensor(config.train.pos_weights, dtype=torch.float32, device=device)

    best_state = copy.deepcopy(model.state_dict())
    best_score = float('inf')
    epochs_without_improvement = 0
    history = []

    for epoch in range(config.train.finetune_epochs):
        train_loss = run_epoch(model, train_loader, optimizer, device, class_weights, pos_weights, scaler, config.train.amp_enabled)
        probs_by_name = predict_probabilities(model, valid_loader, device)
        valid_mae = evaluate_with_default_thresholds(probs_by_name, valid_examples, config.train)
        history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'valid_mae': valid_mae})

        # 这里用“计数后的 MAE”做 early stopping，更贴近最终评测。
        if valid_mae < best_score:
            best_score = valid_mae
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config.train.patience:
                break

    model.load_state_dict(best_state)
    probs_by_name = predict_probabilities(model, valid_loader, device)
    torch.save({'model_state': best_state, 'history': history, 'fold_index': fold_index}, output_dir / f'fold_{fold_index}.pt')
    return best_state, probs_by_name, best_score


def save_oof_predictions(probs_by_name: dict[str, np.ndarray], path: Path) -> None:
    payload = {name: probs.tolist() for name, probs in probs_by_name.items()}
    save_json(payload, path)


def run_training(config: Config) -> Path:
    set_seed(config.train.seed)
    dataset_dir = Path(config.dataset_dir)
    output_dir = Path(config.output_dir)
    clean_artifact_dir(output_dir)

    train_examples, validation_examples, testing_examples = build_examples(dataset_dir)
    hit_bank = build_hit_bank(dataset_dir, train_examples, config.audio)
    background_pool = waveform_pool(train_examples, config.audio.sample_rate)

    manifest = {
        'config': asdict(config),
        'device': str(device),
        'train_examples': len(train_examples),
        'validation_examples': len(validation_examples),
        'testing_examples': len(testing_examples),
        'hit_bank_sizes': {inst: len(items) for inst, items in hit_bank.items()},
    }
    save_json(manifest, output_dir / 'manifest.json')

    initial_state = train_shared_initialization(train_examples, hit_bank, background_pool, config, device)
    splits = split_indices(len(train_examples), config.train.folds, config.train.seed)

    oof_probs = {}
    fold_scores = []
    for fold_index, (train_indices, valid_indices) in enumerate(splits):
        fold_train_examples = [train_examples[index] for index in train_indices]
        fold_valid_examples = [train_examples[index] for index in valid_indices]
        _, fold_probs, fold_score = train_fold(
            fold_index,
            fold_train_examples,
            fold_valid_examples,
            hit_bank,
            background_pool,
            initial_state,
            config,
            device,
            output_dir,
        )
        oof_probs.update(fold_probs)
        fold_scores.append({'fold': fold_index, 'valid_mae': float(fold_score)})

    thresholds = tune_peak_parameters(oof_probs, train_examples, config.train)
    save_json(thresholds, output_dir / 'peak_params.json')
    save_json(fold_scores, output_dir / 'fold_scores.json')
    save_oof_predictions(oof_probs, output_dir / 'oof_probabilities.json')

    oof_rows = []
    for example in train_examples:
        probs = oof_probs[example.stem]
        kd_count = count_from_track(probs[0], float(thresholds['KD']['threshold']), int(thresholds['KD']['min_distance']))
        sd_count = count_from_track(probs[1], float(thresholds['SD']['threshold']), int(thresholds['SD']['min_distance']))
        oof_rows.append({
            'stem': example.stem,
            'label_kd': example.counts['KD'],
            'label_sd': example.counts['SD'],
            'pred_kd': kd_count,
            'pred_sd': sd_count,
        })

    pd.DataFrame(oof_rows).to_csv(output_dir / 'oof_counts.csv', index=False)
    manifest['oof_mae'] = clip_mae([(row['pred_kd'], row['pred_sd']) for row in oof_rows], train_examples)
    save_json(manifest, output_dir / 'manifest.json')
    return output_dir


def load_checkpoints(model_dir: Path, n_mels: int, device: torch.device):
    models = []
    for checkpoint_path in sorted(model_dir.glob('fold_*.pt')):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model = DrumCRNN(n_mels).to(device)
        model.load_state_dict(checkpoint['model_state'])
        model.eval()
        models.append(model)
    if not models:
        raise FileNotFoundError(f'no fold checkpoints found in {model_dir}')
    return models


@torch.no_grad()
def ensemble_predict(models: list[nn.Module], loader: torch.utils.data.DataLoader, device: torch.device):
    aggregated = {}
    counts = {}
    for model in models:
        probs_by_name = predict_probabilities(model, loader, device)
        for name, probs in probs_by_name.items():
            if name not in aggregated:
                aggregated[name] = probs
                counts[name] = 1
            else:
                aggregated[name] += probs
                counts[name] += 1
    for name in aggregated:
        aggregated[name] = aggregated[name] / counts[name]
    return aggregated


def counts_from_probabilities(probs_by_name: dict[str, np.ndarray], examples: list[DrumExample], peak_params: dict[str, dict[str, float | int]]) -> pd.DataFrame:
    rows = []
    for example in examples:
        probs = probs_by_name[example.stem]
        kd_count = count_from_track(probs[0], float(peak_params['KD']['threshold']), int(peak_params['KD']['min_distance']))
        sd_count = count_from_track(probs[1], float(peak_params['SD']['threshold']), int(peak_params['SD']['min_distance']))
        rows.append({'KD': int(kd_count), 'SD': int(sd_count), 'stem': example.stem})
    return pd.DataFrame(rows)


def write_submission_zip(submission_a: Path, submission_b: Path, output_path: Path) -> None:
    with zipfile.ZipFile(output_path, 'w', compression=zipfile.ZIP_DEFLATED) as zip_file:
        zip_file.write(submission_a, arcname='submissionA.csv')
        zip_file.write(submission_b, arcname='submissionB.csv')


def run_prediction(config: Config, model_dir: Path | None = None) -> Path:
    dataset_dir = Path(config.dataset_dir)
    output_dir = Path(config.output_dir)
    artifacts_dir = model_dir or output_dir

    _, validation_examples, testing_examples = build_examples(dataset_dir)
    peak_params = json.loads((artifacts_dir / 'peak_params.json').read_text())
    models = load_checkpoints(artifacts_dir, config.audio.n_mels, device)

    val_loader = torch.utils.data.DataLoader(
        DrumEvalDataset(validation_examples, config.audio),
        batch_size=1,
        shuffle=False,
        num_workers=config.train.num_workers,
        collate_fn=collate_eval,
    )
    test_loader = torch.utils.data.DataLoader(
        DrumEvalDataset(testing_examples, config.audio),
        batch_size=1,
        shuffle=False,
        num_workers=config.train.num_workers,
        collate_fn=collate_eval,
    )

    val_probs = ensemble_predict(models, val_loader, device)
    test_probs = ensemble_predict(models, test_loader, device)

    submission_a_df = counts_from_probabilities(val_probs, validation_examples, peak_params)
    submission_b_df = counts_from_probabilities(test_probs, testing_examples, peak_params)

    submission_a = artifacts_dir / 'submissionA.csv'
    submission_b = artifacts_dir / 'submissionB.csv'
    submission_a_df[['KD', 'SD']].to_csv(submission_a, index=False, header=False)
    submission_b_df[['KD', 'SD']].to_csv(submission_b, index=False, header=False)
    write_submission_zip(submission_a, submission_b, artifacts_dir / 'submission.zip')

    submission_a_df.to_csv(artifacts_dir / 'submissionA_with_stems.csv', index=False)
    submission_b_df.to_csv(artifacts_dir / 'submissionB_with_stems.csv', index=False)
    return artifacts_dir

## 7. 先检查数据和 hit bank，再开始训练

正式训练前，建议先看三个统计量：

- 训练 / 验证 / 测试的样本数。
- `hit bank` 中三类鼓点的片段数。
- 一段短音频转成特征后的形状。

这一步的目的不是调分，而是确认前面的数据处理没有出错。

In [ ]:
train_examples_preview, validation_examples_preview, testing_examples_preview = build_examples(dataset_dir)
hit_bank_preview = build_hit_bank(dataset_dir, train_examples_preview, config.audio)
example_wave = read_audio(train_examples_preview[0].audio_path, config.audio.sample_rate)
example_feature = compute_features(example_wave[: config.audio.sample_rate * 2], config.audio)

print('train / val / test =', len(train_examples_preview), len(validation_examples_preview), len(testing_examples_preview))
print('hit bank sizes =', {key: len(value) for key, value in hit_bank_preview.items()})
print('example feature shape =', example_feature.shape)
print('first train counts =', train_examples_preview[0].counts)

## 8. 正式训练

下面这个 cell 会执行完整的 5 折流程：

- 先用纯合成样本做一个共享初始化。
- 再做 5 折微调。
- 收集 OOF 结果并调峰值参数。

如果你只想先 smoke test，可以把 `folds`、`epochs` 和 `steps_per_epoch` 暂时调小。

In [ ]:
artifacts_dir = run_training(config)
manifest = json.loads((artifacts_dir / 'manifest.json').read_text())
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 9. 生成提交文件

最后一步直接载入各折 checkpoint，把验证集和测试集的概率做折间平均，然后写出：

- `submissionA.csv`
- `submissionB.csv`
- `submission.zip`

其中 zip 包里只包含题目要求的两个 CSV。

In [ ]:
prediction_dir = run_prediction(config)
print('submission path =', prediction_dir / 'submission.zip')
pd.read_csv(prediction_dir / 'submissionA.csv', header=None).head()